In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [3]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [5]:
df.shape

(569, 33)

In [7]:
df.drop(columns=["id","Unnamed: 32"],inplace=True)

In [8]:
df.shape

(569, 31)

In [9]:
df["diagnosis"].value_counts()

,count
diagnosis,
B,357
M,212


In [11]:
x=df.drop(columns=["diagnosis"])
y=df["diagnosis"]

In [12]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [13]:

scaler= StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [14]:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder();
y_train= encoder.fit_transform(y_train)
y_test= encoder.transform(y_test)


###### now,we will start working with pytorch
###### First we need to convert the numpy arrays into tensors to work with pytorch

In [31]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor  = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor  = torch.from_numpy(y_test).float()


In [32]:

X_train_tensor.shape

torch.Size([455, 30])

In [33]:

y_train_tensor.shape

torch.Size([455])

In [34]:
import torch.nn as nn

In [35]:
class MyNeuralNetwork(nn.Module):
  def __init__(self,input_features):
    super().__init__()
    self.linear=nn.Linear(input_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,df):
    output=self.linear(df)
    output=self.sigmoid(output)
    return output


  def loss_function(self,y_pred,y):
    epsilon=1e-7
    y_pred=torch.clamp(y_pred,epsilon,1-epsilon)

    #calculate loss
    loss=-(y*torch.log(y_pred)+(1-y)*torch.log(1-y_pred)).mean()
    return loss

In [36]:
learning_rate=0.1
epochs=25

In [37]:
model = MyNeuralNetwork(X_train_tensor.shape[1])
for epoch in range(epochs):
    # Forward pass
    y_pred = model(X_train_tensor)

    # Compute loss
    loss = model.loss_function(y_pred, y_train_tensor)

    # Backward pass
    loss.backward()

    # Update weights manually
    with torch.no_grad():
        model.linear.weight -= learning_rate * model.linear.weight.grad
        model.linear.bias -= learning_rate * model.linear.bias.grad

        # Zero gradients
        model.linear.weight.grad.zero_()
        model.linear.bias.grad.zero_()

    # Print loss
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')


Epoch 1/25, Loss: 0.7346
Epoch 2/25, Loss: 0.7189
Epoch 3/25, Loss: 0.7100
Epoch 4/25, Loss: 0.7047
Epoch 5/25, Loss: 0.7013
Epoch 6/25, Loss: 0.6988
Epoch 7/25, Loss: 0.6968
Epoch 8/25, Loss: 0.6951
Epoch 9/25, Loss: 0.6936
Epoch 10/25, Loss: 0.6921
Epoch 11/25, Loss: 0.6908
Epoch 12/25, Loss: 0.6895
Epoch 13/25, Loss: 0.6883
Epoch 14/25, Loss: 0.6872
Epoch 15/25, Loss: 0.6861
Epoch 16/25, Loss: 0.6851
Epoch 17/25, Loss: 0.6841
Epoch 18/25, Loss: 0.6832
Epoch 19/25, Loss: 0.6823
Epoch 20/25, Loss: 0.6815
Epoch 21/25, Loss: 0.6807
Epoch 22/25, Loss: 0.6799
Epoch 23/25, Loss: 0.6792
Epoch 24/25, Loss: 0.6785
Epoch 25/25, Loss: 0.6778


In [38]:
model.linear.weight

Parameter containing:
tensor([[-0.0246,  0.1087,  0.0718, -0.1447, -0.0387, -0.1991,  0.1076, -0.1634,
          0.0862,  0.1460,  0.1013, -0.0033, -0.1914,  0.1362, -0.0651,  0.1176,
         -0.0450,  0.1031, -0.0622, -0.0811,  0.0876, -0.1017,  0.1484,  0.0286,
          0.0476,  0.0990, -0.2036, -0.0447, -0.0242,  0.0852]],
       requires_grad=True)

In [39]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.6124961376190186


In [40]:
### instead of usng our own loss fucntion, we will use built in loss fucntion


In [45]:
class MyNeuralNetwork2(nn.Module):
  def __init__(self,input_features):
    super().__init__()
    self.linear=nn.Linear(input_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,df):
    output=self.linear(df)
    output=self.sigmoid(output)
    return output

In [46]:
learning_rate=0.1
epochs=25

In [47]:
loss_function=nn.BCELoss()

In [49]:
model = MyNeuralNetwork2(X_train_tensor.shape[1])
for epoch in range(epochs):
    # Forward pass
    y_pred = model(X_train_tensor)

    # Compute loss
    loss = loss_function(y_pred, y_train_tensor.view(-1,1))

    # Backward pass
    loss.backward()

    # Update weights manually
    with torch.no_grad():
        model.linear.weight -= learning_rate * model.linear.weight.grad
        model.linear.bias -= learning_rate * model.linear.bias.grad

        # Zero gradients
        model.linear.weight.grad.zero_()
        model.linear.bias.grad.zero_()

    # Print loss
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')


Epoch 1/25, Loss: 0.7869
Epoch 2/25, Loss: 0.5655
Epoch 3/25, Loss: 0.4508
Epoch 4/25, Loss: 0.3874
Epoch 5/25, Loss: 0.3472
Epoch 6/25, Loss: 0.3190
Epoch 7/25, Loss: 0.2978
Epoch 8/25, Loss: 0.2810
Epoch 9/25, Loss: 0.2672
Epoch 10/25, Loss: 0.2556
Epoch 11/25, Loss: 0.2457
Epoch 12/25, Loss: 0.2371
Epoch 13/25, Loss: 0.2294
Epoch 14/25, Loss: 0.2226
Epoch 15/25, Loss: 0.2165
Epoch 16/25, Loss: 0.2109
Epoch 17/25, Loss: 0.2058
Epoch 18/25, Loss: 0.2012
Epoch 19/25, Loss: 0.1969
Epoch 20/25, Loss: 0.1929
Epoch 21/25, Loss: 0.1892
Epoch 22/25, Loss: 0.1858
Epoch 23/25, Loss: 0.1826
Epoch 24/25, Loss: 0.1796
Epoch 25/25, Loss: 0.1767


In [ ]:
#lastly, instead up manually updating weights and biases(gd),wewill use inbuilt optimizer


# 🔹 The `torch.optim` Module

`torch.optim` is a module in PyTorch that provides a variety of **optimization algorithms** to update the parameters of your model during training.

It includes **common optimizers** such as:  
- Stochastic Gradient Descent (**SGD**)  
- Adam  
- RMSprop  
- And more  

It also handles weight updates efficiently, including additional features like:  
- Learning rate scheduling  
- Weight decay (regularization)  

---

### 🔹 `model.parameters()` Method

The `model.parameters()` method retrieves an **iterator over all the trainable parameters** (weights and biases) in a model.  

These parameters are instances of `torch.nn.Parameter` and include:

- **Weights**: The weight matrices of layers like `nn.Linear`, `nn.Conv2d`, etc.  
- **Biases**: The bias terms of layers (if they exist).  

The **optimizer** uses these parameters to compute gradients and update them during training.


In [52]:
class MyNeuralNetwork3(nn.Module):
  def __init__(self,input_features):
    super().__init__()
    self.linear=nn.Linear(input_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,df):
    output=self.linear(df)
    output=self.sigmoid(output)
    return output

In [53]:
learning_rate=0.1
epochs=25

In [54]:
loss_function=nn.BCELoss()

In [56]:
model = MyNeuralNetwork3(X_train_tensor.shape[1])

#define optimizer
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)

for epoch in range(epochs):
    # Forward pass
    y_pred = model(X_train_tensor)

    # Compute loss
    loss = loss_function(y_pred, y_train_tensor.view(-1,1))

    #its better to use  zero_grad before  backward pass
    optimizer.zero_grad()
    # Backward pass
    loss.backward()

    #optimization step
    optimizer.step()


    # Print loss
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')


Epoch 1/25, Loss: 0.6970
Epoch 2/25, Loss: 0.5268
Epoch 3/25, Loss: 0.4384
Epoch 4/25, Loss: 0.3849
Epoch 5/25, Loss: 0.3486
Epoch 6/25, Loss: 0.3219
Epoch 7/25, Loss: 0.3013
Epoch 8/25, Loss: 0.2847
Epoch 9/25, Loss: 0.2710
Epoch 10/25, Loss: 0.2594
Epoch 11/25, Loss: 0.2494
Epoch 12/25, Loss: 0.2407
Epoch 13/25, Loss: 0.2331
Epoch 14/25, Loss: 0.2262
Epoch 15/25, Loss: 0.2201
Epoch 16/25, Loss: 0.2145
Epoch 17/25, Loss: 0.2095
Epoch 18/25, Loss: 0.2048
Epoch 19/25, Loss: 0.2006
Epoch 20/25, Loss: 0.1967
Epoch 21/25, Loss: 0.1930
Epoch 22/25, Loss: 0.1896
Epoch 23/25, Loss: 0.1864
Epoch 24/25, Loss: 0.1835
Epoch 25/25, Loss: 0.1807
